**Importing Libraries**

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import requests 
import time
import random
import hashlib
import csv

**EXTRACTING DATA FROM EACH AD FOR HASHING**

In [ ]:
def extract_info_from_ad(ad):
    #extracting title, km, year, price, and fiscal power
    info = {}
    details = ad.find('div', class_='thumb-caption')

    try: #error handling incase details is inexistant
        info['title'] = details.find('h2').get_text(strip=True) if details.find('h2') else '' #we use this structure to avoid crashing
        info['km'] = details.find('li', class_='road').get_text(strip=True) if details.find('li', class_='road') else ''
        info['year'] = details.find('li', class_='year').get_text(strip=True) if details.find('li', class_='year') else ''
        info['power'] = details.find('li', class_='horsepower').get_text(strip=True) if details.find('li', class_='horsepower') else ''
        info['fuel'] = details.find('li', class_='fuel').get_text(strip=True) if details.find('li', class_='fuel') else ''
        info['price'] = ad.find('div', class_='price').get_text(strip=True) if ad.find('div', class_='price') else ''


        return info
    except Exception as e:
        print(f"Error extracting ad info: {e}")
        return None

In [ ]:
def load_seen_hashes(filename='seen_ads.txt'):
    try:
        with open(filename, 'r') as f:
            return set(f.read().splitlines())
    except FileNotFoundError:
        return set()

def save_seen_hashes(hashes, filename='seen_ads.txt'):
    with open(filename, 'w') as f:
        for h in hashes:
            f.write(h + '\n')

In [ ]:
seen_ad_hashes = load_seen_hashes()

def hash_ad_content(ad_info):

    key = f"{ad_info['title']}-{ad_info['price']}-{ad_info['km']}-{ad_info['year']}"
    return hashlib.md5(key.encode()).hexdigest()

In [ ]:
def is_duplicate_ad(ad_data):

    ad_hash = hash_ad_content(ad_data)

    if ad_hash in seen_ad_hashes:
        return True
    seen_ad_hashes.add(ad_hash)
    return False

**ADS' LINK EXTRACTION FUNCTION**

In [ ]:
base_url = "https://www.automobile.tn"

def extract_links_from_page(soup, page):
    try: #exception handling for missing ads (html elements)
        ads = soup.find('div', class_='articles')
        if not ads:
            print(f"No ads found on {page}")
            return []
    except Exception as e:
        print(f"Error parsing ads on page {page}: {e}")
        return [] #we return an empty list so the scraper can continue running
    all_ads = ads.find_all('div', class_='occasion-item-v2') #identifying the exact div for each ad, no need for every nested div

    links = []
    duplicates_found = 0

    if all_ads:
        for ad in all_ads: 
            a_tag = ad.find('a', class_='occasion-link-overlay')
            ad_data = extract_info_from_ad(ad)
            if a_tag and ad_data: 
                if not is_duplicate_ad(ad_data):
                    link = base_url + a_tag['href']
                    links.append(link)
                else:
                    duplicates_found += 1
                    print(f" Duplicate AD: {ad_data.get('title')[:30]}...")
            else:
                print('ad found without valid link')
    else:
        print('Could not find ads container')
        
    print(f"  → Page {page}: {len(links)} unique ads, {duplicates_found} duplicates filtered")
    return links

**ALL LINKS EXTRACTION FUNCTION**

In [ ]:
def scrape_all_pages():
    all_links = []
    page = 1
    last_page = 10
    while page <= last_page:
        print(f"Scraping page {page}...")
        url = f"https://www.automobile.tn/fr/occasion/s=/{page}"
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status() #Raises a terminal ERROR if the response is not 200.
        except requests.exceptions.RequestException as e:
            print(f"Request failed on page {page}: {e}")
            time.sleep(3)
            continue
        soup = BeautifulSoup(response.text, 'lxml')

        pagination = soup.find('ul', class_='pagination')
        if pagination: #if pagination doesn't show handle exception
            pages = pagination.find_all('a')
            numeric_pages = [a for a in pages if a.text.isdigit()]
            if numeric_pages:
                last_page = int(numeric_pages[-1].text) #get the last cell page number
        
        page_links = extract_links_from_page(soup, page)
        all_links.extend(page_links)

        page += 1
        time.sleep(random.uniform(1,3))

    return all_links

* Both **network errors** and **parsing errors** are being handled
* I can know exactly what's happening during scraping, through the logging system
* Automated web scraper that systematically collects data, self managing,

**File creation**

In [ ]:
def save_links_to_csv(links, filename="links.csv"):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["link"])  # header
        for link in links:
            writer.writerow([link])

In [ ]:
all_links = scrape_all_pages()
save_links_to_csv(all_links, 'car_links.csv')
save_seen_hashes(seen_ad_hashes)